# LLM Architecture
Learning Goals

Description
In this lab we will be building out the architecture of a large language model (LLM).

Citation
Raschka, S. (2024). Build a large language model (from scratch). Manning Publications.

Lab Deliverables
Read though: https://github.com/rasbt/LLM-workshop-2024/blob/main/03_architecture/03.ipynb

# Step 1. Coding an LLM architecture
- Set a all capital variable GPT_CONFIG_124M
  -  variables in all capitals are conventially seen as constants but python techincally doesn't have constants, so it's more a conventions saying "don't change this"
- Set the value to that variable to a dictionary with the folowing values
  - "vocab_size": 50257,    # Vocabulary size
  - "context_length": 1024, # Context length
  - "emb_dim": 768,         # Embedding dimension
  - "n_heads": 12,          # Number of attention heads
  - "n_layers": 12,         # Number of layers
  - "drop_rate": 0.0,       # Dropout rate
  - "qkv_bias": False       # Query-Key-Value bias

  <details>
  <summary>Click Here to view solution</summary>
 ```python
  GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
  }
 ```
   </details>



In [3]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    
    "context_length": 1024, 
    "emb_dim": 768,         
    "n_heads": 12,          
    "n_layers": 12,         
    "drop_rate": 0.0,      
    "qkv_bias": False      
}

GPT_CONFIG_124M


{'vocab_size': 50257,
 'context_length': 1024,
 'emb_dim': 768,
 'n_heads': 12,
 'n_layers': 12,
 'drop_rate': 0.0,
 'qkv_bias': False}

## Step 2: Build the GPT Model


- Import torch.nn as nn
- Import TransformerBlock and LayerNorm
- Create a class called GPTModel
- The class should inherit from nn.Module
- The constructor (__init__) should take a single argument called cfg
- Inside __init__, define the following attributes:
  - Token embedding  
    - tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
  - Positional embedding  
    - pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
  - Embedding dropout  
    - drop_emb = nn.Dropout(cfg["drop_rate"])
  - Transformer blocks  
    - trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
  - Final layer normalization  
    - final_norm = LayerNorm(cfg["emb_dim"])
  - Output head  
    - out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

- Create a method called forward that takes one argument called in_idx.
  - Inside this method:
    - Extract batch_size and seq_len from in_idx.shape  
    - Create token embeddings using self.tok_emb(in_idx)  
    - Create positional embeddings using torch.arange(seq_len, device=in_idx.device)  
    - Add token and positional embeddings together  
    - Pass the result through drop_emb  
    - Pass the result through trf_blocks  
    - Pass the result through final_norm  
    - Compute logits using out_head  
    - Return logits  


<details>
  <summary>Click Here to view solution</summary>

    ```
    import torch
    import torch.nn as nn
    from supplementary import TransformerBlock, LayerNorm


    class GPTModel(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
            self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
            self.drop_emb = nn.Dropout(cfg["drop_rate"])

            self.trf_blocks = nn.Sequential(
                *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
            )

            self.final_norm = LayerNorm(cfg["emb_dim"])
            self.out_head = nn.Linear(
                cfg["emb_dim"], cfg["vocab_size"], bias=False
            )

        def forward(self, in_idx):
            batch_size, seq_len = in_idx.shape
            tok_embeds = self.tok_emb(in_idx)
            pos_embeds = self.pos_emb(
                torch.arange(seq_len, device=in_idx.device)
            )
            x = tok_embeds + pos_embeds
            x = self.drop_emb(x)
            x = self.trf_blocks(x)
            x = self.final_norm(x)
            logits = self.out_head(x)
            return logits
    ```
</details>

In [4]:
import torch
import torch.nn as nn
from supplementary import TransformerBlock, LayerNorm


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

## Step 3: Instantiate the GPT Model with Weights

- Import torch and tiktoken
- Set a variable called tokenizer using tiktoken.get_encoding("gpt2")
- Create an empty list called batch
- Create two strings of text and assign them to variables txt1 and txt2
- Encode each string using the tokenizer
- Convert each encoded sequence to a torch tensor
- Append both tensors to the batch list
- Stack the batch into a single tensor using torch.stack with dim=0
- Print the batch
- Set a manual random seed using torch.manual_seed(123)
- Instantiate the GPTModel using the configuration GPT_CONFIG_124M
- Pass the batch into the model and store the result in a variable called out
- Print the input batch
- Print the shape of the output
- Print the output tensor



<details>
  <summary>Click Here to view solution</summary>
    ```python
    import torch
    import tiktoken

    tokenizer = tiktoken.get_encoding("gpt2")

    batch = []

    txt1 = "Every effort moves you"
    txt2 = "Every day holds a"

    batch.append(torch.tensor(tokenizer.encode(txt1)))
    batch.append(torch.tensor(tokenizer.encode(txt2)))

    batch = torch.stack(batch, dim=0)
    print(batch)

    torch.manual_seed(123)
    model = GPTModel(GPT_CONFIG_124M)

    out = model(batch)

    print("Input batch:\n", batch)
    print("\nOutput shape:", out.shape)
    print(out)
    ```


In [5]:
import torch
import tiktoken

torch.set_printoptions(edgeitems=3, threshold=20)

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))

batch = torch.stack(batch, dim=0)
print(batch)

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(batch)

print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print("\nOutput tensor:\n", out)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])
Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])

Output tensor:
 tensor([[[ 6.4164e-02,  2.0443e-01, -1.6945e-01,  ...,  1.7887e-01,
           2.1921e-01, -5.8153e-01],
         [ 3.7736e-01, -4.2545e-01, -6.5874e-01,  ..., -2.5050e-01,
           4.6553e-01, -2.5760e-01],
         [ 8.8996e-01, -1.3770e-01,  1.4748e-01,  ...,  1.7770e-01,
          -1.2015e-01, -1.8902e-01],
         [-9.7276e-01,  9.7338e-02, -2.5419e-01,  ...,  1.1035e+00,
           3.7639e-01, -5.9006e-01]],

        [[ 6.4164e-02,  2.0443e-01, -1.6945e-01,  ...,  1.7887e-01,
           2.1921e-01, -5.8153e-01],
         [ 1.3433e-01, -2.1289e-01, -2.7020e-02,  ...,  8.1153e-01,
          -4.7410e-02,  3.1186e-01],
         [ 8.9996e-01,  9.5396e-01, -1.7896e-01,  ...,  8.3053e-01,
           2.7657e-01, -2.4577e-02],
         [-9.2894e-05,  1.9390e-01,  5.1217e-01,  ...

# Exercise: Generate some text
  Note: No solution given for this
  1. Use the tokenizer.encode method to prepare some input text
  2. Then, convert this text into a pytprch tensor via (torch.tensor)
  3. Add a batch dimension via .unsqueeze(0)
  4. Use the generate_text_simple function (Provided below) to have the GPT generate some text based on your prepared input text
  5. The output from step 4 will be token IDs, convert them back into text via the tokenizer.decode method

In [6]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [ ]:

start_text = "Once upon a time"

token_ids = tokenizer.encode(start_text)
input_tensor = torch.tensor(token_ids)
input_tensor = input_tensor.unsqueeze(0)
output_ids = generate_text_simple(
    model=model,
    idx=input_tensor,
    max_new_tokens=20,
    context_size=128
)
generated_text = tokenizer.decode(output_ids.squeeze(0).tolist())
print(generated_text)

Once upon a time schedules sarcastallo CNNmaybeuto bodies Illinois intervenarf CLIdisplay marched barric sponsoringSecretary instinctively fe limitation gallery
